In [ ]:
import os
import pandas as pd, matplotlib.pyplot as plt, numpy as np
import mne

# sklearn
import sklearn
import sklearn.model_selection

from pypdf import PdfReader
import glob


print(os.getenv("KAGGLE_API_TOKEN") is not None)

True


In [52]:
!git clone https://github.com/linq-rag/FinanceRAG.git

Cloning into 'FinanceRAG'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 57 (delta 8), reused 2 (delta 2), pack-reused 40 (from 1)
Receiving objects: 100% (57/57), 32.20 KiB | 499.00 KiB/s, done.
Resolving deltas: 100% (11/11), done.


In [53]:
DATA_PATH = '/workspaces/llm_finetuning/FinanceRAG'

ACM-ICAIF '24 FinanceRAG Challenge

The Financial RAG Challenge aims to advance Retrieval-Augmented Generation (RAG) systems that can efficiently handle lengthy and complex financial documents. Participants are tasked with building systems capable of retrieving relevant contexts from large financial datasets and generating precise answers to financial queries, addressing real-world challenges such as financial terminology, industry-specific language, and numerical data.

For the Task 1, evaluation of retrieval process will use the NDCG@10 (Normalized Discounted Cumulative Gain) metric. NDCG is a commonly used metric in information retrieval to evaluate the ranking quality of search results. It measures how well the predicted ranking of relevant documents matches the ideal ranking. NDCG@10 specifically evaluates the top 10 results returned for each query.

#### Load data

In [41]:
pdf_path = os.path.join(DATA_PATH, 'Structured data-20250319T105519Z-001')
print(f'Number of pdf files: {len(glob.glob(os.path.join(pdf_path, "**", "*.pdf")))}')

pdf_files = glob.glob(os.path.join(pdf_path, "**", "*.pdf"))
for i,el in enumerate(pdf_files):
    reader = PdfReader(el)
    print(f"{i}: {el.split('/')[-1]} - Number of pages: {len(reader.pages)}")

Number of pdf files: 0


In [32]:
csv_path = os.path.join(DATA_PATH, 'Data_ret.csv')
qa_data = pd.read_csv(csv_path, index_col=0)
qa_data 

,Question,Context,Value,prompt
0,the value of B-BBEE level (South Africa) in Ab...,• The company is committed to economic sustai...,1.0,"[INST] Given the question and context, extract..."
1,the value of B-BBEE level (South Africa) in Ab...,• The company is committed to economic sustai...,2.0,"[INST] Given the question and context, extract..."
2,the value of B-BBEE level (South Africa) in Ab...,• The company is committed to economic sustai...,1.0,"[INST] Given the question and context, extract..."
3,the value of B-BBEE level (South Africa) in Ab...,• The company is committed to economic sustai...,1.0,"[INST] Given the question and context, extract..."
4,the value of Board meeting attendance (%) in A...,non-executive directors||11|12|11|12|\n|Non-e...,95.0,"[INST] Given the question and context, extract..."
...,...,...,...,...
1912,the value of Waste recycled in Oceana1&2 docum...,"264|16,081|\n\n## WATER\n\n|Total volume of wa...",2110.0,"[INST] Given the question and context, extract..."
1913,the value of Waste recycled in Oceana1&2 docum...,"264|16,081|\n\n## WATER\n\n|Total volume of wa...",2986.0,"[INST] Given the question and context, extract..."
1914,the value of Absolute GHG emissions (Total sco...,"Scope 1, 2 and 3 emissions – market-based|484 ...",302045.0,"[INST] Given the question and context, extract..."
1915,the value of Absolute GHG emissions (Total sco...,"Scope 1, 2 and 3 emissions – market-based|484 ...",312753.0,"[INST] Given the question and context, extract..."


##### Check missing values, duplicates, column types

In [4]:
# Check train_df 
print(train_df_raw.columns)

print(f'Duplicates:')
print(train_df_raw[train_df_raw.duplicated()])

train_df_no_dup = train_df_raw.drop_duplicates(keep='first')

print(f'Number of raws original df: {train_df_raw.shape[0]} - After removing duplicates: {train_df_no_dup.shape[0]}')

# Missing values 
# drops raws containing at least one missing value
train_df = train_df_no_dup.dropna()
print(f'\nNumber of raws train df after removing missing values: {train_df.shape[0]}')

Index(['eeg_id', 'eeg_sub_id', 'eeg_label_offset_seconds', 'spectrogram_id',
       'spectrogram_sub_id', 'spectrogram_label_offset_seconds', 'label_id',
       'patient_id', 'expert_consensus', 'seizure_vote', 'lpd_vote',
       'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote'],
      dtype='str')
Duplicates:
Empty DataFrame
Columns: [eeg_id, eeg_sub_id, eeg_label_offset_seconds, spectrogram_id, spectrogram_sub_id, spectrogram_label_offset_seconds, label_id, patient_id, expert_consensus, seizure_vote, lpd_vote, gpd_vote, lrda_vote, grda_vote, other_vote]
Index: []
Number of raws original df: 106800 - After removing duplicates: 106800

Number of raws train df after removing missing values: 106800


In [5]:
train_df.head(5)

,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0


In [6]:
len(train_df['eeg_id'].unique())

17089

In [7]:
# Check test_df 
print(test_df_raw.columns)

print(f'Duplicates:\n')
print(test_df_raw[test_df_raw.duplicated()])

test_df_no_dup = test_df_raw.drop_duplicates(keep='first')

print(f'Number of raws original df: {test_df_raw.shape[0]} - After removing duplicates: {test_df_no_dup.shape[0]}')

# Missing values 
test_df = test_df_no_dup.dropna()
print(f'Number of raws test df after removing missing values: {test_df.shape[0]}')

Index(['spectrogram_id', 'eeg_id', 'patient_id'], dtype='str')
Duplicates:

Empty DataFrame
Columns: [spectrogram_id, eeg_id, patient_id]
Index: []
Number of raws original df: 1 - After removing duplicates: 1
Number of raws test df after removing missing values: 1


In [8]:
len(test_df['eeg_id'].unique())

1

In [9]:
test_df

,spectrogram_id,eeg_id,patient_id
0,853520,3911565283,6885


In [10]:
# check  sample_submission_raw
sample_submission_raw

,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667


In [11]:
# Create Target
Target = {'Seizure':0, 'LPD':1, 'GPD':2, 'LRDA':3, 'GRDA':4, 'Other':5}

#### From EEG to spectrogram

In [16]:
specs_ids = os.listdir(os.path.join(DATA_PATH, "train_spectrograms"))
eegs_ids = os.listdir(os.path.join(DATA_PATH, "train_eegs"))

print(list(set(specs_ids) & set(eegs_ids)))

[]


In [17]:
# single id example
spec_id = train_df["spectrogram_id"].unique()[0] 

# corresponding eeg_id (first one, eeg_sub_id = 0)
eeg_id = train_df[train_df['spectrogram_id']==spec_id]['eeg_id'][0]

spec_path = f"{DATA_PATH}/train_spectrograms/{spec_id}.parquet"
spec = pd.read_parquet(spec_path)

eeg_path = f"{DATA_PATH}/train_eegs/{eeg_id}.parquet"
eeg = pd.read_parquet(eeg_path)

In [18]:
spec

,time,LL_0.59,LL_0.78,LL_0.98,LL_1.17,LL_1.37,LL_1.56,LL_1.76,LL_1.95,LL_2.15,...,RP_18.16,RP_18.36,RP_18.55,RP_18.75,RP_18.95,RP_19.14,RP_19.34,RP_19.53,RP_19.73,RP_19.92
0,1,4.26,10.98,9.05,13.65,11.49,8.930000,18.840000,19.26,19.240000,...,0.31,0.17,0.28,0.19,0.24,0.27,0.29,0.16,0.22,0.19
1,3,2.65,3.97,12.18,13.26,14.21,13.230000,9.650000,8.11,11.280000,...,0.15,0.13,0.14,0.24,0.24,0.36,0.35,0.31,0.36,0.40
2,5,4.18,4.53,8.77,14.26,13.36,16.559999,19.219999,17.51,22.650000,...,0.29,0.21,0.16,0.25,0.28,0.28,0.34,0.48,0.44,0.48
3,7,2.41,3.21,4.92,8.07,5.97,12.420000,10.820000,14.96,21.809999,...,0.33,0.51,0.49,0.64,0.58,0.42,0.32,0.31,0.32,0.33
4,9,2.29,2.44,2.77,4.62,5.39,7.080000,9.840000,12.27,14.410000,...,0.44,0.38,0.48,0.63,0.45,0.45,0.49,0.33,0.31,0.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,631,6.36,6.59,6.60,7.30,4.48,8.400000,13.420000,13.85,16.010000,...,0.14,0.05,0.06,0.04,0.04,0.04,0.05,0.05,0.08,0.11
316,633,4.90,8.80,8.22,5.83,10.21,10.580000,10.250000,13.68,19.549999,...,0.16,0.08,0.06,0.06,0.07,0.04,0.06,0.09,0.07,0.08
317,635,6.07,7.85,11.26,9.20,8.18,9.130000,10.450000,15.09,23.020000,...,0.15,0.13,0.13,0.13,0.10,0.08,0.07,0.09,0.17,0.12
318,637,3.41,3.75,4.80,6.45,6.70,7.960000,8.160000,6.97,9.700000,...,0.13,0.11,0.13,0.07,0.11,0.12,0.09,0.16,0.19,0.19


In [19]:
train_df.head(4)

,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0


In [20]:
# single id example
spec_id = train_df["spectrogram_id"].unique()[0] 

# corresponding eeg_id (first one, eeg_sub_id = 0)
eeg_id = train_df[train_df['spectrogram_id']==spec_id]['eeg_id'][0]

spec_path = f"{DATA_PATH}/train_spectrograms/{spec_id}.parquet"
spec = pd.read_parquet(spec_path)

eeg_path = f"{DATA_PATH}/train_eegs/{eeg_id}.parquet"
eeg = pd.read_parquet(eeg_path)

In [21]:
eeg

,Fp1,F3,C3,P3,F7,T3,T5,O1,Fz,Cz,Pz,Fp2,F4,C4,P4,F8,T4,T6,O2,EKG
0,-80.519997,-70.540001,-80.110001,-108.750000,-120.330002,-88.620003,-101.750000,-104.489998,-99.129997,-90.389999,-97.040001,-77.989998,-88.830002,-112.120003,-108.110001,-95.949997,-98.360001,-121.730003,-106.449997,7.920000
1,-80.449997,-70.330002,-81.760002,-107.669998,-120.769997,-90.820000,-104.260002,-99.730003,-99.070000,-92.290001,-96.019997,-84.500000,-84.989998,-115.610001,-103.860001,-97.470001,-89.290001,-115.500000,-102.059998,29.219999
2,-80.209999,-75.870003,-82.050003,-106.010002,-117.500000,-87.489998,-99.589996,-96.820000,-119.680000,-99.360001,-91.110001,-99.440002,-104.589996,-127.529999,-113.349998,-95.870003,-96.019997,-123.879997,-105.790001,45.740002
3,-84.709999,-75.339996,-87.480003,-108.970001,-121.410004,-94.750000,-105.370003,-100.279999,-113.839996,-102.059998,-95.040001,-99.230003,-101.220001,-125.769997,-111.889999,-97.459999,-97.180000,-128.940002,-109.889999,83.870003
4,-90.570000,-80.790001,-93.000000,-113.870003,-129.960007,-102.860001,-118.599998,-101.099998,-107.660004,-102.339996,-98.510002,-95.300003,-88.930000,-115.639999,-99.800003,-97.500000,-88.730003,-114.849998,-100.250000,97.769997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17995,-144.660004,-147.809998,-129.820007,-129.460007,-157.509995,-124.000000,-124.570000,-94.820000,-153.070007,-121.110001,-86.459999,-132.520004,-138.339996,-128.970001,-71.300003,-114.480003,-86.709999,-114.959999,-81.500000,-20.070000
17996,-140.880005,-153.000000,-129.529999,-129.020004,-154.059998,-131.220001,-128.380005,-95.000000,-140.820007,-114.639999,-84.379997,-115.339996,-119.230003,-114.709999,-70.989998,-92.129997,-79.639999,-116.139999,-81.879997,10.600000
17997,-133.729996,-141.770004,-121.900002,-122.370003,-158.750000,-123.550003,-127.730003,-93.089996,-125.230003,-106.489998,-83.419998,-112.720001,-103.209999,-107.629997,-61.869999,-97.910004,-77.150002,-106.500000,-75.339996,-2.060000
17998,-141.449997,-151.139999,-127.190002,-128.699997,-163.460007,-124.309998,-129.479996,-94.419998,-140.869995,-113.339996,-83.519997,-129.300003,-118.650002,-117.589996,-71.879997,-99.279999,-83.900002,-116.160004,-81.410004,2.820000


EEG COlumns (20) are the channels = 19 electrodes EEG + 1 EGK channel (battito cardiaco, registrato per controllo).

Rows are the temporal instants, the samples in the time.

Each row is a measure of the electrodes in a precise time instant.

Il sengla è campionato a 200Hz (200 misurazioni al secondo per ogni canale) --> 1800 righe / 200 Hz = 90 secondi

In [22]:
import joblib 
from tqdm import tqdm

In [23]:
# Define a function to process a single eeg_id
def process_spec(spec_id, split="train"):
    if not os.path.exists(f"{DATA_PATH}/{split}_spectrograms/{spec_id}.npy"):
        spec_path = f"{DATA_PATH}/{split}_spectrograms/{spec_id}.parquet"
        spec = pd.read_parquet(spec_path)
        spec = spec.fillna(0).values[:, 1:].T # fill NaN values with 0, transpose for (Time, Freq) -> (Freq, Time)
        spec = spec.astype("float32")
        np.save(f"{DATA_PATH}/{split}_spectrograms/{spec_id}.npy", spec)

# Get unique spec_ids of train and valid data
spec_ids = train_df["spectrogram_id"].unique()

# Parallelize the processing using joblib for training data
_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(process_spec)(spec_id, "train")
    for spec_id in tqdm(spec_ids, total=len(spec_ids))
)

# Get unique spec_ids of test data
test_spec_ids = test_df["spectrogram_id"].unique()

# Parallelize the processing using joblib for test data
_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(process_spec)(spec_id, "test")
    for spec_id in tqdm(test_spec_ids, total=len(test_spec_ids)))

  0%|          | 0/11138 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 562.31it/s]


#### DataLoader

- Read the npy spectrogram files
- Extract labeled subsamples using specified offset values

In [24]:
train = train_df.groupby('eeg_id')[['spectrogram_id','spectrogram_label_offset_seconds']]

train = train_df.groupby('eeg_id')[['spectrogram_id','spectrogram_label_offset_seconds']].agg(
    {'spectrogram_id':'first','spectrogram_label_offset_seconds':'min'})
train.columns = ['spec_id','min']

tmp = train_df.groupby('eeg_id')[['spectrogram_id','spectrogram_label_offset_seconds']].agg(
    {'spectrogram_label_offset_seconds':'max'})
train['max'] = tmp

TARGETS = train_df.columns[-6:]
TARGETS

tmp = train_df.groupby('eeg_id')[TARGETS].agg('sum')
for t in TARGETS:
    train[t] = tmp[t].values

y_data = train[TARGETS].values
y_data = y_data / y_data.sum(axis=1,keepdims=True)
train[TARGETS] = y_data

In [25]:
train

,spec_id,min,max,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
eeg_id,,,,,,,,,
568657,789577333,0.0,16.0,0.0,0.000000,0.25,0.000000,0.166667,0.583333
582999,1552638400,0.0,38.0,0.0,0.857143,0.00,0.071429,0.000000,0.071429
642382,14960202,1008.0,1032.0,0.0,0.000000,0.00,0.000000,0.000000,1.000000
751790,618728447,908.0,908.0,0.0,0.000000,1.00,0.000000,0.000000,0.000000
778705,52296320,0.0,0.0,0.0,0.000000,0.00,0.000000,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...
4293354003,1188113564,0.0,0.0,0.0,0.000000,0.00,0.000000,0.500000,0.500000
4293843368,1549502620,0.0,0.0,0.0,0.000000,0.00,0.000000,0.500000,0.500000
4294455489,2105480289,0.0,0.0,0.0,0.000000,0.00,0.000000,0.000000,1.000000


In [26]:
tmp = train_df.groupby('eeg_id')[['expert_consensus']].agg('first')
train['target'] = tmp

In [27]:
train = train.reset_index()
print('Train non-overlapp eeg_id shape:', train.shape )
train.head()

Train non-overlapp eeg_id shape: (17089, 11)


,eeg_id,spec_id,min,max,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,target
0,568657,789577333,0.0,16.0,0.0,0.000000,0.25,0.000000,0.166667,0.583333,Other
1,582999,1552638400,0.0,38.0,0.0,0.857143,0.00,0.071429,0.000000,0.071429,LPD
2,642382,14960202,1008.0,1032.0,0.0,0.000000,0.00,0.000000,0.000000,1.000000,Other
3,751790,618728447,908.0,908.0,0.0,0.000000,1.00,0.000000,0.000000,0.000000,GPD
4,778705,52296320,0.0,0.0,0.0,0.000000,0.00,0.000000,0.000000,1.000000,Other


In [28]:
y_data = train[TARGETS].values
y_data.shape

(17089, 6)

In [29]:
from torch.utils.data import Dataset, DataLoader

In [30]:
class EEGSpectrogramDataset(Dataset):

    def __init__(self, df, indices, y_data):
        self.df = df.iloc[indices].reset_index(drop=True)
        self.y = y_data[indices]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        spec_id = self.df.iloc[idx]["spec_id"]

        spec = np.load(
            f"{DATA_PATH}/train_spectrograms/{spec_id}.npy"
        )

        # Keep first 300 time bins
        spec = spec[:, :300]

        # Average every 5 time bins
        # (400, 300) -> (400, 60)
        spec = spec.reshape(400, 60, 5).mean(axis=2)

        spec = torch.from_numpy(spec).float().unsqueeze(0)

        label = torch.tensor(
            self.y[idx],
            dtype=torch.float32
        )

        return spec, label

In [31]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [32]:
import torch
import torch.nn as nn

In [33]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


for fold, (train_idx, val_idx) in enumerate(
    skf.split(train, train["target"])
):

    print(f"\n========== FOLD {fold + 1}/5 ==========")

    train_dataset = EEGSpectrogramDataset(
        train,
        train_idx,
        y_data
    )

    val_dataset = EEGSpectrogramDataset(
        train,
        val_idx,
        y_data
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    print("Train samples:", len(train_dataset))
    print("Validation samples:", len(val_dataset))

Device: cpu

========== FOLD 1/5 ==========
Train samples: 13671
Validation samples: 3418

========== FOLD 2/5 ==========
Train samples: 13671
Validation samples: 3418

========== FOLD 3/5 ==========
Train samples: 13671
Validation samples: 3418

========== FOLD 4/5 ==========
Train samples: 13671
Validation samples: 3418

========== FOLD 5/5 ==========
Train samples: 13672
Validation samples: 3417


In [34]:
import torch
import torch.nn as nn


class EEGCNN(nn.Module):
    def __init__(self, n_classes=6):
        super().__init__()

        self.features = nn.Sequential(
            # 1 x 400 x 308
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 32 x 200 x 154
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 64 x 100 x 77
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 128 x 50 x 38
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            # indipendente dalla dimensione esatta dello spettrogramma
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [37]:
model = EEGCNN(n_classes=6)

x = torch.randn(8, 1, 400, 60)

output = model(x)

print(output.shape)

torch.Size([8, 6])


In [38]:
import torch
from torch.utils.data import TensorDataset, DataLoader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [39]:
fold_results = []

In [40]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


for fold, (train_idx, val_idx) in enumerate(
    skf.split(train, train["target"])
):

    print(f"\n========== FOLD {fold + 1}/5 ==========")

    train_dataset = EEGSpectrogramDataset(
        train,
        train_idx,
        y_data
    )

    val_dataset = EEGSpectrogramDataset(
        train,
        val_idx,
        y_data
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    print("Train samples:", len(train_dataset))
    print("Validation samples:", len(val_dataset))

    # -------------------------
    # New model for each fold
    # -------------------------
    model = EEGCNN(n_classes=6).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    criterion = torch.nn.KLDivLoss(
        reduction="batchmean"
    )

    # -------------------------
    # Epochs
    # -------------------------
    n_epochs = 20

    for epoch in range(n_epochs):

        # =====================
        # TRAIN
        # =====================
        model.train()

        train_loss = 0.0

        for X_batch, y_batch in tqdm(
            train_loader,
            desc=f"Fold {fold+1} | Epoch {epoch+1}/{n_epochs} | Train"
        ):

            X_batch = X_batch.to(
                device,
                non_blocking=True
            )

            y_batch = y_batch.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            output = model(X_batch)

            log_probs = torch.log_softmax(
                output,
                dim=1
            )

            loss = criterion(
                log_probs,
                y_batch
            )

            loss.backward()

            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)

        train_loss /= len(train_loader.dataset)

        # =====================
        # VALIDATION
        # =====================
        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(
                    device,
                    non_blocking=True
                )

                y_batch = y_batch.to(
                    device,
                    non_blocking=True
                )

                output = model(X_batch)

                log_probs = torch.log_softmax(
                    output,
                    dim=1
                )

                loss = criterion(
                    log_probs,
                    y_batch
                )

                val_loss += loss.item() * X_batch.size(0)

        val_loss /= len(val_loader.dataset)

        print(
            f"Epoch {epoch+1:02d}/{n_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )

    fold_results.append(val_loss)

Device: cpu

========== FOLD 1/5 ==========
Train samples: 13671
Validation samples: 3418


Fold 1 | Epoch 1/20 | Train:   0%|          | 0/214 [00:00<?, ?it/s]/usr/local/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Fold 1 | Epoch 1/20 | Train:  65%|██████▍   | 139/214 [10:14<05:31,  4.42s/it]


KeyboardInterrupt: 

In [ ]:
print("\nFold validation losses:")
print(fold_results)

print(
    f"Mean validation loss: "
    f"{np.mean(fold_results):.4f}"
)

print(
    f"Std validation loss: "
    f"{np.std(fold_results):.4f}"
)